In [1]:
# Want to recreate the problem
from util import load_ensemble

In [2]:
ens_file = "/pscratch/sd/j/jkalloor/bqskit/block_checkpoints_nisq_0/adder9_1_3.5_250/ensemble_0__try1.qasms"

In [3]:
circs = load_ensemble(ens_file)

SPlit String


Decoded


In [4]:
circ_0 = circs[0]
circ_0.num_params

198

In [ ]:
from bqskit.ir.circuit import Circuit, CircuitPoint
from bqskit.ir.gates import CNOTGate, U3Gate
from bqskit.passes import ToU3Pass
flooded_cnot = Circuit(2)
flooded_cnot.append_gate(CNOTGate(), (0, 1))
flooded_cnot.append_gate(U3Gate(), (0,), [0, 0, 0])
flooded_cnot.append_gate(U3Gate(), (1,), [0, 0, 0])
def flood_circ(c: Circuit):
    circ = c.copy()
    init_u3_circ = Circuit(circ.num_qudits)
    for i in range(circ.num_qudits):
        init_u3_circ.append_gate(U3Gate(), (i,), [0, 0, 0])
    circ.insert_circuit(0, init_u3_circ, tuple(range(circ.num_qudits)))
    for cycle, op in circ.operations_with_cycles():
        if isinstance(op.gate, CNOTGate):
            op_pt = CircuitPoint(cycle, op.location[0])
            circ.replace_with_circuit(op_pt, flooded_cnot.copy(), as_circuit_gate=True)
    circ.unfold_all()
    ToU3Pass.run_group_circ(circ)
    return circ

old_param_nums = [c.num_params for c in circs[:100]]
new_circs = [flood_circ(c) for c in circs[:100]]
new_param_nums = [c.num_params for c in new_circs]

In [13]:
def delete_random_gates(c: Circuit):
    import random
    circ = c.copy()
    pts = []
    for cycle, op in circ.operations_with_cycles(reverse=True):
        pt = CircuitPoint(cycle, op.location[0])
        if random.random() < 0.3:
            pts.append(pt)
    circ.batch_pop(pts)
    return circ

new_circs_2 = [delete_random_gates(c) for c in new_circs]
new_param_nums_2 = [c.num_params for c in new_circs_2]

In [14]:
print(old_param_nums[20:50])
print(new_param_nums[20:50])
print(new_param_nums_2[20:50])


[192, 192, 192, 204, 192, 192, 192, 192, 192, 186, 192, 186, 192, 192, 192, 198, 198, 192, 210, 192, 210, 210, 210, 192, 198, 192, 198, 210, 192, 192]
[192, 192, 192, 204, 192, 192, 192, 192, 192, 186, 192, 186, 192, 192, 192, 198, 198, 192, 210, 192, 210, 210, 210, 192, 198, 192, 198, 210, 192, 192]
[144, 138, 156, 150, 138, 141, 144, 144, 135, 132, 138, 132, 117, 123, 132, 144, 141, 159, 156, 153, 132, 138, 156, 141, 117, 153, 156, 147, 144, 135]


In [15]:
import numpy as np
from bqskit.ir.lang import get_language
from util import normalized_gp_frob_cost

lang = get_language('qasm')

rand_ind = 6
cir = circs[rand_ind]
flood_cir = new_circs[rand_ind]

un_1 = cir.get_unitary()
un_2 = flood_cir.get_unitary()
print(normalized_gp_frob_cost(un_1, un_2))
un_3 = flood_cir.get_unitary(cir.params)
print(normalized_gp_frob_cost(un_1, un_3))

post_qasm_cir_1 = lang.decode(lang.encode(cir))
post_qasm_cir_2 = lang.decode(lang.encode(flood_cir))

un_4 = post_qasm_cir_1.get_unitary(cir.params)
un_5 = post_qasm_cir_2.get_unitary(cir.params)
print(normalized_gp_frob_cost(un_1, un_4))
print(normalized_gp_frob_cost(un_1, un_5))

5.912466771060311e-16
0.0
0.0
0.0


In [22]:
cir = new_circs_2[rand_ind]
un_1 = cir.get_unitary()

init_params = cir.params[:15]

post_qasm_cir_1 = lang.decode(lang.encode(cir))
final_params = post_qasm_cir_1.params[:15]
post_qasm_cir_2 = lang.decode(lang.encode(post_qasm_cir_1))
final_params_2 = post_qasm_cir_2.params[:15]

un_4 = post_qasm_cir_1.get_unitary(cir.params)
un_5 = post_qasm_cir_1.get_unitary()
un_6 = post_qasm_cir_2.get_unitary(post_qasm_cir_1.params)

mod_params = 

np.set_printoptions(precision=3, threshold=np.inf, linewidth=np.inf)

print(init_params)
print(final_params)
print(final_params_2)

print(normalized_gp_frob_cost(un_1, un_4))
print(normalized_gp_frob_cost(un_1, un_5))
print(normalized_gp_frob_cost(un_5, un_6))

[ 1.312e-08  2.608e+00 -2.538e+00  3.142e+00 -1.167e+00  1.403e+00  3.142e+00  3.372e+00 -2.815e-01  1.571e+00 -4.253e-01  4.687e-01  3.142e+00  2.815e-01  2.815e-01]
[ 1.312e-08  2.608e+00 -2.538e+00  3.142e+00 -1.167e+00  1.403e+00  3.142e+00  3.372e+00 -2.815e-01  1.571e+00 -4.253e-01  4.687e-01  1.571e+00 -1.571e+00  2.127e-01]
[ 1.312e-08  2.608e+00 -2.538e+00  3.142e+00 -1.167e+00  1.403e+00  3.142e+00  3.372e+00 -2.815e-01  1.571e+00 -4.253e-01  4.687e-01  1.571e+00 -1.571e+00  2.127e-01]
0.9991014151188061
4.518950394777445e-16
0.0


(250, 80, 246)


In [6]:
params = jiggle_array[0][0]
print(params.shape)

(246,)


In [27]:
# Get target un
from util import load_block
from bqskit.ir import Circuit

target_circ = Circuit.from_file(load_block("adder9", 1))
target_un = target_circ.get_unitary()

In [28]:
circ.set_params(params)
un1 = circ.get_unitary()

In [29]:
params_chopped = params[:circ.num_params]
circ.set_params(params_chopped)
un2 = circ.get_unitary()

In [30]:
# un1.get_distance_from(target_un)

np.float64(0.9999994166193984)

In [32]:
from util import normalized_gp_frob_cost
from bqskit.ir.opt.cost.functions import GPNormalizedFrobeniusCostGenerator
frob_cost_gen = GPNormalizedFrobeniusCostGenerator()
frob_cost = frob_cost_gen.gen_cost(circ, target_un)

dists = [frob_cost.get_cost(params) for params in jiggle_array[0]]
# dists =[normalized_gp_frob_cost(circ.get_unitary(params), un2) for params in jiggle_array[0]]
print(np.mean(dists), np.std(dists))

0.9994588830699817 4.315318841229094e-06


In [56]:
def get_a_dist(ind: int, offset: int = 0) -> float:
    circ = circs[ind]
    # target_un = circ.get_unitary(jiggle_array[ind][0])
    frob_cost = frob_cost_gen.gen_cost(circ, target_un)
    params = jiggle_array[ind][offset]
    dist = frob_cost.get_cost(params)
    return dist

In [47]:
print(get_a_dist(1), get_a_dist(2), get_a_dist(3))
print([c.num_params for c in circs[1:4]])

0.0009993118171214717 0.0008989099314480716 0.8361689802085657
[246, 192, 192]


In [61]:
from util import store_jiggled_ensemble
circ_inds = list(range(10, 200, 3))
test_ens_file = "test_ens.qasms"
test_jiggle_file = "test_jiggles.npy"
circ_params = [(circs[j], jiggle_array[j]) for j in circ_inds]
store_jiggled_ensemble(circ_params, test_ens_file, test_jiggle_file)

In [62]:
from util import load_jiggled_ensemble
test_circs = load_jiggled_ensemble(test_ens_file, test_jiggle_file, use_mp=True)

SPlit String
Decoded
Num Circs:  64
Params Shape:  (64, 80, 246)


In [64]:
offset = 25
avg_dists = [get_a_dist(i, offset) for i in circ_inds]
dists = [frob_cost_gen.calc_cost(circ, target_un) for circ in test_circs[offset:len(test_circs):80]]
# print(np.mean(dists))
# print(dists)
print(np.array(dists) / np.array(avg_dists))

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [1]:
# Testing Gridsynth
import random
from pygridsynth.gridsynth import gridsynth_gates
from bqskit.ir import Circuit
from bqskit.ir.gates import *
from util import normalized_gp_frob_cost

def gridsynth_gates_to_cir(gates: str):
    circ = Circuit(1)
    for gate in gates:
        if gate == 'I':
            circ.append_gate(IdentityGate(), (0,))
        elif gate == 'Z':
            circ.append_gate(ZGate(), (0,))
        elif gate == 'S':
            circ.append_gate(SGate(), (0,))
        elif gate == 'L':
            circ.append_gate(SdgGate(), (0,))
        elif gate == 'T':
            circ.append_gate(TGate(), (0,))
        elif gate == "H":
            circ.append_gate(HGate(), (0,))
        elif gate == "D":
            circ.append_gate(TdgGate(), (0,))
        elif gate == "X":
            circ.append_gate(XGate(), (0,))
    return circ


def remove_t(s):
    indices = [i for i, c in enumerate(s) if c == 'T']
    if not indices:
        return s  # No 'T' to remove

    remove_idx = random.choice(indices)
    return s[:remove_idx] + s[remove_idx + 1:]

def delete_all_possible_ts(angle: float, epsilon: float):
    orig_str = gridsynth_gates(angle, epsilon ** 2)
    num_fails = 0
    num_ts_removed = 0
    rz = RZGate()
    target = rz.get_unitary([angle])
    orig_grid_circ = gridsynth_gates_to_cir(orig_str)
    orig_cost = normalized_gp_frob_cost(target, orig_grid_circ.get_unitary())
    assert orig_cost < epsilon
    f = orig_str
    while num_fails < 50:                
        new_grid_str = remove_t(f)
        new_grid_circ = gridsynth_gates_to_cir(new_grid_str)
        new_cost = normalized_gp_frob_cost(target, new_grid_circ.get_unitary())
        if new_cost < epsilon:
            f = new_grid_str
            num_ts_removed += 1
        else:
            num_fails += 1 
    return num_ts_removed

In [20]:
import numpy as np
all_angles = np.random.uniform(0, 2 * np.pi, 20)
epsilons = [1e-1, 1e-2, 1e-3]
for epsilon in epsilons:
    num_ts_removed = [delete_all_possible_ts(angle, epsilon) for angle in all_angles]
    print(np.mean(num_ts_removed))

KeyboardInterrupt: 

In [ ]:

# Run grid_synth multiple times on the same angle and see how many distinct strings it creates
angle = np.random.uniform(0, 2 * np.pi)
num_trials = 25
epsilons = [1e-1, 1e-2, 1e-3, 5e-4]
gridsynth_strs = {}
for epsilon in epsilons:
    num_strs = []
    gridsynth_strs[epsilon] = {}
    for angle in all_angles[:4]:
        grid_strs = set()
        target = RZGate().get_unitary([angle])
        for _ in range(num_trials):
            gates = gridsynth_gates(angle, epsilon ** 2)
            grid_circ = gridsynth_gates_to_cir(gates)
            cost = normalized_gp_frob_cost(target, grid_circ.get_unitary())
            assert cost < epsilon
            grid_strs.add(gridsynth_gates(angle, epsilon ** 2))
        gridsynth_strs[epsilon][angle] = grid_strs
        num_strs.append(len(grid_strs))

    print(epsilon ** 2, np.mean(num_strs))

NameError: name 'all_angles' is not defined

In [18]:
# Calculate the bias reduction of each of these sets
from util import normalized_gp_frob_cost
from bqskit.qis import UnitaryMatrix

def fix_phase(circuit: Circuit, target: UnitaryMatrix) -> tuple[float, float]:
    unitary = circuit.get_unitary()
    global_phase_correction = target.get_target_correction_factor(unitary)
    circuit.append_gate(GlobalPhaseGate(1, global_phase=global_phase_correction), (0,))

epsilons = [1e-1, 1e-2, 1e-3, 5e-4]

for epsilon in epsilons:
    for angle in all_angles[:3]:
        target = RZGate().get_unitary([angle])
        grid_strs = gridsynth_strs[epsilon][angle]
        grid_circs = [gridsynth_gates_to_cir(g) for g in grid_strs]
        [fix_phase(c, target) for c in grid_circs]
        grid_unitaries = [np.array(c.get_unitary()) for c in grid_circs]
        # Correct global phase
        costs = [normalized_gp_frob_cost(un, target) for un in grid_unitaries]
        mean_un = np.mean(grid_unitaries, axis=0)
        avg_cost = np.mean(costs)
        cost_of_avg = normalized_gp_frob_cost(mean_un, target)
        print("Avg: Epsilon: ", avg_cost, " Mean Unitary Distance: ", cost_of_avg)

NameError: name 'all_angles' is not defined

In [5]:
# Now test if adding perturbations to a given RZ angle can reduce the bias

starting_angle = np.random.uniform(0, 2 * np.pi)
target = RZGate().get_unitary([starting_angle])

# Perturb angle by a small amount
epsilones = [1e-1, 1e-2, 1e-3, 1e-4]
for epsilon in epsilones:
    perturbed_angles = [starting_angle + np.random.uniform(- 5 * epsilon, 5 * epsilon) for _ in range(100)]

    perturbed_unitaries = [RZGate().get_unitary([angle]) for angle in perturbed_angles]

    perturbed_costs = [normalized_gp_frob_cost(un, target) for un in perturbed_unitaries]

    mean_un = np.mean(perturbed_unitaries, axis=0)

    cost_of_mean = normalized_gp_frob_cost(mean_un, target)
    ratio = cost_of_mean / np.mean(perturbed_costs) / np.mean(perturbed_costs)

    print("Avg: Epsilon: ", np.mean(perturbed_costs), " Cost of Mean Unitary: ", cost_of_mean, "Ratio: ", ratio)



Avg: Epsilon:  0.07969112273271606  Cost of Mean Unitary:  0.011926505749990993 Ratio:  1.8779902404646245
Avg: Epsilon:  0.009311120747669637  Cost of Mean Unitary:  0.00023902307984569 Ratio:  2.756994633261266
Avg: Epsilon:  0.0009284826790216216  Cost of Mean Unitary:  9.544785522706414e-05 Ratio:  110.71808392388637
Avg: Epsilon:  9.245905569577003e-05  Cost of Mean Unitary:  5.911550779345401e-06 Ratio:  691.5164525539541


In [6]:
# Testing pauli twirling
import scipy as sp
import numpy as np
starting_angle = np.random.uniform(0, 2 * np.pi)
target = RZGate().get_unitary([starting_angle])

pauli_gates = [ZGate(), SGate(), SdgGate()]
pauli_uns = [gate.get_unitary() for gate in pauli_gates]

# Perturb angle by a small amount
epsilones = [1e-1, 1e-2, 1e-3, 1e-4]
for epsilon in epsilones:
    pos_pauli_perturbations = [sp.linalg.expm(1j * epsilon * un) for un in pauli_uns]
    neg_pauli_perturbations = [sp.linalg.expm(-1j * epsilon * un) for un in pauli_uns]
    pauli_perturbations = pos_pauli_perturbations + neg_pauli_perturbations
    perturbed_unitaries = [g @ target @ g.conj().T for g in pauli_perturbations]
    perturbed_costs = [normalized_gp_frob_cost(un, target) for un in perturbed_unitaries]

    mean_un = np.mean(perturbed_unitaries, axis=0)

    cost_of_mean = normalized_gp_frob_cost(mean_un, target)
    ratio = cost_of_mean / np.mean(perturbed_costs) / np.mean(perturbed_costs)

    print("Avg: Epsilon: ", np.mean(perturbed_costs), " Cost of Mean Unitary: ", cost_of_mean, "Ratio: ", ratio)

Avg: Epsilon:  0.06711200084703141  Cost of Mean Unitary:  0.006688918539691937 Ratio:  1.4850994362711536
Avg: Epsilon:  0.006667111120000054  Cost of Mean Unitary:  6.666888891851354e-05 Ratio:  1.4998500099993304
Avg: Epsilon:  0.0006666671111112248  Cost of Mean Unitary:  6.666668889226574e-07 Ratio:  1.4999985000768008
Avg: Epsilon:  6.666666711114198e-05  Cost of Mean Unitary:  6.666666777153075e-09 Ratio:  1.5000000048580528


In [26]:
# Now test Pauli Twirling with gridsynth
def t_count(circ: Circuit):
    return circ.count(TGate()) + circ.count(TdgGate())

starting_angle = np.random.uniform(0, 2 * np.pi)
target = RZGate().get_unitary([starting_angle])

angle_perturbations = np.array([-1, 1, -np.pi, np.pi, -np.pi/2, np.pi/2])

# Perturb angle by a small amount
epsilones = [1e-1, 1e-2, 1e-3, 5e-4]
for epsilon in epsilones:
    orig_t_str = gridsynth_gates(starting_angle, epsilon ** 2)
    orig_t_circ = gridsynth_gates_to_cir(orig_t_str)
    orig_t_count = t_count(orig_t_circ)
    # pos_pauli_perturbations = [sp.linalg.expm(1j * epsilon * un) for un in pauli_uns]
    # neg_pauli_perturbations = [sp.linalg.expm(-1j * epsilon * un) for un in pauli_uns]
    # pauli_perturbations = pos_pauli_perturbations + neg_pauli_perturbations
    # perturbed_unitaries = [g @ target @ g.conj().T for g in pauli_perturbations]
    # print(perturbed_unitaries)
    # perturbed_angles = [RZGate.calc_params(un) for un in perturbed_unitaries]
    perturbed_angles = epsilon * angle_perturbations + starting_angle
    # print("Starting Angle: ", starting_angle)
    # print("Angles: ", perturbed_angles)
    perturb_t_strs = [gridsynth_gates(an, (epsilon ** 2) * 10) for an in perturbed_angles]
    perturbed_t_circs = [gridsynth_gates_to_cir(t) for t in perturb_t_strs]
    [fix_phase(c, target) for c in perturbed_t_circs]
    perturbed_unitaries = [c.get_unitary() for c in perturbed_t_circs]
    perturbed_t_counts = [t_count(c) for c in perturbed_t_circs]

    perturbed_costs = [normalized_gp_frob_cost(un, target) for un in perturbed_unitaries]

    mean_un = np.mean(perturbed_unitaries, axis=0)

    cost_of_mean = normalized_gp_frob_cost(mean_un, target)
    ratio = cost_of_mean / np.mean(perturbed_costs) / np.mean(perturbed_costs)

    print("Avg: Epsilon: ", np.mean(perturbed_costs), " Mean Unitary Distance: ", cost_of_mean, "Ratio: ", ratio)
    print("Orig T Count: ", orig_t_count, "Avg T Count: ", np.mean(perturbed_t_counts))

Avg: Epsilon:  0.07041167281896116  Mean Unitary Distance:  0.014298530817003757 Ratio:  2.8840453900857206
Orig T Count:  20 Avg T Count:  8.666666666666666
Avg: Epsilon:  0.006615374701938362  Mean Unitary Distance:  0.00020953622374682705 Ratio:  4.787956725961787
Orig T Count:  40 Avg T Count:  29.666666666666668
Avg: Epsilon:  0.0006729019924272573  Mean Unitary Distance:  2.4790066859246422e-06 Ratio:  5.474873255459716
Orig T Count:  62 Avg T Count:  50.666666666666664
Avg: Epsilon:  0.00033720244007890294  Mean Unitary Distance:  2.1407570036096006e-07 Ratio:  1.8827209544062744
Orig T Count:  66 Avg T Count:  55.666666666666664
